# Dogs vs. Cats: Exploratory Data Analysis

This is the first of three notebooks in a compact transfer-learning case study on
Kaggle's **Dogs vs. Cats** dataset. Before training anything, it is worth establishing
what the data actually looks like, not as a ritual but because two specific properties
of this dataset directly determine choices made in the next notebook.

Unlike MNIST, where every image is a centered 28x28 grayscale digit, these are 25,000
real photographs scraped from the web. They vary in resolution, aspect ratio, subject
scale, lighting, and how much of the frame the animal actually occupies. That variation
is the whole reason a pretrained backbone is the right tool here.

We will:

* Confirm the class balance rather than trusting the competition description.
* Look at raw images from both classes, to see the range of poses and framing we are asking a model to handle.
* Survey image dimensions and aspect ratios, which motivates the resize/crop strategy in the augmentation pipeline.
* Produce a stratified train/validation split, saved to disk so the training and evaluation notebooks provably share the same partition.

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from catsdogs.dataset import class_balance, label_from_filename, list_image_paths, stratified_split

DATA_DIR = Path("../data/raw/train")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Class balance

The Kaggle training set is labeled by filename (`cat.0.jpg`, `dog.0.jpg`, ...) rather
than by folder, which is why `catsdogs.dataset` derives labels from the filename prefix
instead of using torchvision's `ImageFolder`.

The competition advertises an even 12,500/12,500 split. It costs one cell to verify
that, and the answer matters: a balanced dataset means plain accuracy is a defensible
headline metric, and it means a stratified split is a cheap safeguard rather than a
necessity.

In [ ]:
paths = list_image_paths(DATA_DIR)
counts = class_balance(paths)
print(f"{len(paths)} images total: {counts}")

plt.bar(counts.keys(), counts.values())
plt.ylabel("count")
plt.title("Class balance")
plt.show()

## Sample images

A handful of images from each class. The point of this cell is not to confirm that cats
look like cats. It is to calibrate expectations about difficulty. Look for how often the
animal is partially occluded, off-center, small in frame, photographed alongside a
human, or lit badly. Those are the cases the error analysis in notebook 03 will return
to.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, class_name in enumerate(["cat", "dog"]):
    examples = [p for p in paths if p.name.startswith(class_name)][:6]
    for col, path in enumerate(examples):
        axes[row, col].imshow(Image.open(path))
        axes[row, col].set_title(path.name, fontsize=8)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## Image dimensions and aspect ratio

This is the survey that actually drives a modeling decision. A pretrained ImageNet
backbone expects a fixed 224x224 input, so every image has to be resized, but *how*
depends on the shape of the distribution below.

If aspect ratios cluster tightly near 1.0, a naive resize distorts little and is fine.
The wider the spread, the more a plain `Resize((224, 224))` squashes subjects, and the
more a crop-based strategy is worth preferring. Sampling every 20th image is plenty to
characterize the distribution without opening 25,000 files.

In [ ]:
sample = paths[::20]  # every 20th image is enough to characterize the distribution
sizes = [Image.open(p).size for p in sample]
df_sizes = pd.DataFrame(sizes, columns=["width", "height"])

(df_sizes["width"] / df_sizes["height"]).plot(
    kind="hist", bins=30, title="Aspect ratio (width / height)"
)
plt.xlabel("width / height")
plt.show()

df_sizes.describe()

The takeaway feeds directly into `build_transforms` in `src/catsdogs/dataset.py`:
training uses `RandomResizedCrop`, which samples a random 80-100% region and resizes it
to 224x224. That doubles as augmentation and as a principled answer to non-square
inputs, since it crops rather than squashing. Validation uses a plain deterministic
resize, because evaluation must not be stochastic.

## Train/validation split

An 85/15 stratified split. Stratification matters less on a balanced dataset than it
would on a skewed one, but it is nearly free and it removes sampling noise as an
explanation for any class asymmetry we see in the confusion matrix later.

The split is written to `data/processed/` rather than recomputed per notebook. Both
later notebooks read these CSVs, so the model is never evaluated on an image it trained
on. That is a guarantee a shared random seed alone does not give you, since it silently
breaks the moment someone changes a split parameter in one notebook and not the other.

In [ ]:
train_paths, train_labels, val_paths, val_labels = stratified_split(
    paths, val_fraction=0.15, seed=0
)
print(f"train: {len(train_paths)}, val: {len(val_paths)}")
print(f"val class balance: {class_balance(val_paths)}")

pd.DataFrame({"path": [str(p) for p in train_paths], "label": train_labels}).to_csv(
    PROCESSED_DIR / "train_split.csv", index=False
)
pd.DataFrame({"path": [str(p) for p in val_paths], "label": val_labels}).to_csv(
    PROCESSED_DIR / "val_split.csv", index=False
)

## Closing notes

Two properties of this dataset shape everything that follows. It is **balanced**, so
accuracy is a meaningful headline number and the confusion matrix should be roughly
symmetric if the model is behaving. And it is **visually heterogeneous** (varied
resolution, framing, and occlusion), which is exactly the regime where fine-tuning
pretrained ImageNet features beats training a small CNN from scratch, since low-level
edge and texture detectors transfer directly and do not need to be relearned from 21,000
photographs.

Notebook 02 fine-tunes a pretrained ResNet18 on the split produced above.